# Notebook 2: LLaVA-Style Projector — Single-Stage SFT on VizWiz-LF

**Architecture:** Frozen CLIP-ViT-Large-Patch14 + Frozen Phi-3-mini-4k + Trainable 2-layer MLP Projector  
**Training:** Train ONLY the MLP Projector. Projector upcast to FP32 to prevent PyTorch GradScaler ValueError.  
**Data:** Single-file `LF.json` — synthetic answers for training, expert answers held out for evaluation.  
**Environment:** Kaggle Dual-T4 (2×16 GB)


## 0. Install Dependencies

In [1]:
!pip install -q --upgrade \
    transformers \
    accelerate \
    datasets \
    huggingface_hub \
    evaluate \
    rouge_score \
    nltk \
    bert_score \
    sacrebleu \
    sentencepiece \
    bitsandbytes \
    opencv-python-headless

## 1. Imports & Global Configuration

In [ ]:
import os
import json
import shutil
import glob
import logging
import random
import csv
from pathlib import Path
from typing import Optional, List, Dict, Any

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Modern AMP imports — replaces deprecated torch.cuda.amp
from torch.amp import GradScaler, autocast

from transformers import (
    CLIPVisionModel,
    CLIPImageProcessor,
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    TrainerState,
    TrainerControl,
)
from huggingface_hub import HfApi, login

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
LF_JSON_PATH      = "/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf/LF.json"
TRAIN_IMG_DIR     = "/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf/train/train"
VAL_IMG_DIR       = "/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf/val/val"
OUTPUT_DIR        = "/kaggle/working/llava_projector_sft"
EXPERT_HOLDOUT_PATH = "/kaggle/working/expert_holdout.json"
LOSS_CSV_PATH     = "/kaggle/working/train_loss_log.csv"
HUB_MODEL_ID      = "Abdullah-Khan-Niazi/vizwiz-lf-llava-projector"  # ← change me
HF_TOKEN          = ''                # set in Kaggle Secrets

# ── Hyperparameters ───────────────────────────────────────────────────────────
CLIP_MODEL_ID  = "openai/clip-vit-large-patch14"
PHI3_MODEL_ID  = "microsoft/Phi-3-mini-4k-instruct"
PROJ_HIDDEN    = 2048           # hidden dim of MLP projector
MAX_SEQ_LEN    = 512
IMG_SIZE       = 224            # resize target for CLIP
BATCH_SIZE     = 4
GRAD_ACCUM     = 8
NUM_EPOCHS     = 3
LR             = 2e-4
WARMUP_STEPS   = 50             # replaces deprecated warmup_ratio
NUM_WORKERS    = 2

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
logger.info("Configuration loaded. CUDA available: %s", torch.cuda.is_available())
logger.info("GPU count: %d", torch.cuda.device_count())

2026-05-02 07:01:28,831 INFO Configuration loaded. CUDA available: True
2026-05-02 07:01:28,863 INFO GPU count: 2


## 2. High-Speed Image Path Indexer  
**CRITICAL:** Build a RAM dictionary once — never call `os.path.exists()` inside `__getitem__`.

In [3]:
def build_image_path_map(*image_dirs: str) -> Dict[str, str]:
    """
    Walk each image directory once and build a filename→full_path RAM dictionary.
    Covers both train/train/ and val/val/ Kaggle layout.
    """
    path_map: Dict[str, str] = {}
    for img_dir in image_dirs:
        if not os.path.isdir(img_dir):
            logger.warning("Image directory not found: %s", img_dir)
            continue
        for fpath in glob.iglob(os.path.join(img_dir, "**", "*"), recursive=True):
            if fpath.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                fname = os.path.basename(fpath)
                path_map[fname] = fpath
    logger.info("IMAGE_PATH_MAP built: %d entries", len(path_map))
    return path_map

IMAGE_PATH_MAP = build_image_path_map(TRAIN_IMG_DIR, VAL_IMG_DIR)

2026-05-02 07:02:34,927 INFO IMAGE_PATH_MAP built: 31704 entries


## 3. Parse `LF.json` — Synthetic Train Split & Expert Holdout

In [4]:
def parse_lf_json(lf_path: str) -> tuple[list, list]:
    """
    Parse LF.json and route answers:
      - source contains 'expert' or 'human'  → expert_holdout (held out, not trained on)
      - all others (GPT-4V, LLaVA, etc.)     → synthetic_train
    """
    with open(lf_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    synthetic_train: list = []
    expert_holdout:  list = []

    # LF.json structure: list of items OR dict keyed by image_id
    items = data if isinstance(data, list) else list(data.values())

    for item in items:
        # --- FIX: Safely extract filename from image_url ---
        img_url = item.get("image_url", item.get("image", ""))
        image_name = os.path.basename(img_url) if img_url else ""
        
        question   = item.get("question", "").strip()
        long_answers = item.get("long_answers", {})

        # long_answers may be a dict {source: answer_text} or list of dicts
        if isinstance(long_answers, dict):
            answer_items = long_answers.items()
        elif isinstance(long_answers, list):
            answer_items = [(a.get("source", ""), a.get("answer", "")) for a in long_answers]
        else:
            continue

        for source, answer_data in answer_items:
            # --- FIX: Extract text from nested answer_paragraph ---
            if isinstance(answer_data, dict):
                answer_text = answer_data.get("answer_paragraph", "")
            else:
                answer_text = str(answer_data)
                
            if not answer_text or not answer_text.strip():
                continue
                
            record = {
                "image":    image_name,
                "question": question,
                "answer":   answer_text.strip(),
                "source":   str(source),
            }
            src_lower = str(source).lower()
            if "expert" in src_lower or "human" in src_lower:
                expert_holdout.append(record)
            else:
                synthetic_train.append(record)

    logger.info("Parsed LF.json → synthetic_train: %d | expert_holdout: %d",
                len(synthetic_train), len(expert_holdout))
    return synthetic_train, expert_holdout


synthetic_train, expert_holdout = parse_lf_json(LF_JSON_PATH)

# ── Immediately save expert holdout to disk ───────────────────────────────────
with open(EXPERT_HOLDOUT_PATH, "w", encoding="utf-8") as f:
    json.dump(expert_holdout, f, indent=2, ensure_ascii=False)
logger.info("Expert holdout saved to: %s", EXPERT_HOLDOUT_PATH)

# Sanity-check a sample
print("\n=== Synthetic Train Sample ===")
print(json.dumps(synthetic_train[0], indent=2) if synthetic_train else "EMPTY")
print("\n=== Expert Holdout Sample ===")
print(json.dumps(expert_holdout[0], indent=2) if expert_holdout else "EMPTY")

2026-05-02 07:02:35,029 INFO Parsed LF.json → synthetic_train: 3596 | expert_holdout: 600
2026-05-02 07:02:35,036 INFO Expert holdout saved to: /kaggle/working/expert_holdout.json



=== Synthetic Train Sample ===
{
  "image": "VizWiz_train_00014902.jpg",
  "question": "I'm trying to determine what this jar is. It looks like pasta sauce, but I can't tell. I got a mystery bag of groceries in the grocery delivery yesterday and I'm trying to figure out what these things are.",
  "answer": "The image you've provided is quite blurry and the details are not clear, which makes it difficult to identify the jar's contents with certainty. However, the color scheme suggests it might be a jar of pasta sauce or something similar due to the shades of red and yellow visible, which are common colors for pasta sauce packaging.\nFor future reference, if you're trying to determine the contents of a package, you can sometimes feel for braille labels or use a smartphone app that can read text aloud from images which might be helpful for identifying items in a mystery grocery bag. If you have any other items or clearer images, feel free to share them, and I'll do my best to assist you.

In [5]:
import os
import torch
from PIL import Image
from tqdm.auto import tqdm

# We will save the pre-computed CLIP tensors here
TENSOR_DIR = "/kaggle/working/clip_tensors"
os.makedirs(TENSOR_DIR, exist_ok=True)

logger.info("Pre-computing CLIP image tensors to eliminate CPU bottleneck...")

for idx, item in enumerate(tqdm(synthetic_train, desc="Processing Images")):
    save_path = os.path.join(TENSOR_DIR, f"img_{idx}.pt")
    
    if os.path.exists(save_path):
        continue
        
    img_path = IMAGE_PATH_MAP.get(item["image"])
    
    try:
        image = Image.open(img_path).convert("RGB")
        # clip_processor handles the resizing and normalization
        pixel_values = clip_processor(images=image, return_tensors="pt")["pixel_values"][0]
        
        # Cast to FP16 to keep the file sizes tiny!
        torch.save(pixel_values.to(torch.float16), save_path)
    except Exception as e:
        logger.error(f"Failed on {item['image']}: {e}")
        # Save a blank tensor as a safe fallback so training doesn't crash
        torch.save(torch.zeros((3, 224, 224), dtype=torch.float16), save_path)

logger.info(f"✅ CLIP Preprocessing complete! Saved to {TENSOR_DIR}")

2026-05-02 07:02:35,045 INFO Pre-computing CLIP image tensors to eliminate CPU bottleneck...


Processing Images:   0%|          | 0/3596 [00:00<?, ?it/s]

2026-05-02 07:02:35,077 INFO ✅ CLIP Preprocessing complete! Saved to /kaggle/working/clip_tensors


## 4. PyTorch Dataset

In [6]:
class VizWizLFDataset(Dataset):
    """
    Loads (image, question, answer) triples for the LLaVA-style projector.

    Images are:
      1. Looked up via IMAGE_PATH_MAP (no disk searching inside __getitem__).
      2. Loaded with cv2 + resized to IMG_SIZE×IMG_SIZE immediately to cap VRAM.
      3. Converted to PIL.Image for CLIPImageProcessor compatibility.
    """

    PROMPT_TEMPLATE = (
        "<image>\n"
        "Question: {question}\n"
        "Answer: {answer}"
    )
    def __init__(
        self,
        records: list,
        clip_processor: CLIPImageProcessor,
        tokenizer,
        image_path_map: dict,
        max_seq_len: int = MAX_SEQ_LEN,
    ):
        self.records        = records
        self.clip_processor = clip_processor
        self.tokenizer      = tokenizer
        self.image_path_map = image_path_map
        self.max_seq_len    = max_seq_len

        # Filter records whose image is not in the map
        before = len(self.records)
        self.records = [
            r for r in self.records
            if os.path.basename(r["image"]) in self.image_path_map
        ]
        after = len(self.records)
        if before != after:
            logger.warning("Dropped %d records with missing images.", before - after)

    def __len__(self):
        return len(self.records)

    def _load_image(self, image_name: str) -> Image.Image:
        """Fast image load: cv2 read → resize → RGB PIL.Image."""
        fpath = self.image_path_map[os.path.basename(image_name)]
        img_bgr = cv2.imread(fpath)
        if img_bgr is None:
            # Fallback: black image
            img_bgr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img_rgb)

    def __getitem__(self, idx) -> Dict[str, torch.Tensor]:
        item = self.records[idx]
        
        # ── 1. LIGHTNING FAST IMAGE LOADING ──────────────────────────────
        tensor_path = os.path.join("/kaggle/working/clip_tensors", f"img_{idx}.pt")
        
        # Load the FP16 tensor and cast it to FP32/BF16 based on what your model wants
        pixel_values = torch.load(tensor_path, weights_only=True).to(torch.float32)

        # ── 2. TEXT TOKENIZATION (Keep this as-is) ───────────────────────
        # Text: instruction + answer as one string; labels mask the instruction
        instruction = f"Question: {item['question']}\nAnswer: "
        full_text = instruction + item["answer"]

        instruction_ids = self.tokenizer(
            instruction, add_special_tokens=False
        ).input_ids
        full_ids = self.tokenizer(
            full_text,
            add_special_tokens=True,
            max_length=self.max_seq_len,
            truncation=True,
        ).input_ids

        input_ids = torch.tensor(full_ids, dtype=torch.long)
        labels    = input_ids.clone()
        # Mask instruction tokens with -100 (only train on answer tokens)
        n_inst = min(len(instruction_ids), len(full_ids))
        labels[:n_inst] = -100

        attention_mask = torch.ones_like(input_ids)

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

## 5. Collator — Dynamic Padding

In [7]:
def llava_data_collator(features: List[Dict[str, torch.Tensor]], pad_token_id: int = 0):
    """
    Dynamically pads input_ids / attention_mask / labels to the longest sequence
    in the batch. pixel_values are already fixed-size from CLIP processor.
    """
    pixel_values   = torch.stack([f["pixel_values"]   for f in features], dim=0)
    max_len        = max(f["input_ids"].size(0) for f in features)

    input_ids_padded      = []
    attention_mask_padded = []
    labels_padded         = []

    for f in features:
        seq_len = f["input_ids"].size(0)
        pad_len = max_len - seq_len

        input_ids_padded.append(
            torch.cat([f["input_ids"],
                       torch.full((pad_len,), pad_token_id, dtype=torch.long)])
        )
        attention_mask_padded.append(
            torch.cat([f["attention_mask"],
                       torch.zeros(pad_len, dtype=torch.long)])
        )
        labels_padded.append(
            torch.cat([f["labels"],
                       torch.full((pad_len,), -100, dtype=torch.long)])
        )

    return {
        "pixel_values":   pixel_values,
        "input_ids":      torch.stack(input_ids_padded,      dim=0),
        "attention_mask": torch.stack(attention_mask_padded, dim=0),
        "labels":         torch.stack(labels_padded,         dim=0),
    }

## 6. Model Architecture — LLaVA-Style Projector

### Critical Cross-Device Fix
Because `device_map='auto'` places Phi-3 embeddings on CPU and the Projector on GPU,  
we **dynamically look up the embedding layer device** at forward-time and move tensors accordingly.

In [8]:
class MLPProjector(nn.Module):
    """
    2-layer MLP that projects CLIP visual features into Phi-3's embedding space.
    Upcast to FP32 to prevent GradScaler ValueError when the LLM runs in BF16.
    """

    def __init__(self, clip_hidden: int, llm_hidden: int, proj_hidden: int = PROJ_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(clip_hidden, proj_hidden),
            nn.GELU(),
            nn.Linear(proj_hidden, llm_hidden),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.float())  # upcast input to FP32 for stability


class LLaVAStyleModel(nn.Module):
    """
    End-to-end LLaVA-style model:
      - Frozen CLIP-ViT-Large-Patch14 vision encoder
      - Frozen Phi-3-mini-4k language model
      - Trainable 2-layer MLP projector (FP32)
    """

    def __init__(
        self,
        clip_model_id: str = CLIP_MODEL_ID,
        phi3_model_id: str = PHI3_MODEL_ID,
        proj_hidden:   int  = PROJ_HIDDEN,
    ):
        super().__init__()
        
        self.is_model_parallel = True

        logger.info("Loading CLIP vision encoder: %s", clip_model_id)
        self.clip = CLIPVisionModel.from_pretrained(
            clip_model_id, torch_dtype=torch.float16
        ).to("cuda:0")
        for p in self.clip.parameters():
            p.requires_grad_(False)
            
        # <--- FIX 1: Enable Gradient Checkpointing on CLIP
        self.clip.gradient_checkpointing_enable()

        logger.info("Loading Phi-3 LLM: %s", phi3_model_id)
        self.phi3 = AutoModelForCausalLM.from_pretrained(
            phi3_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",          
            trust_remote_code=False,    
        )
        for p in self.phi3.parameters():
            p.requires_grad_(False)

        # <--- FIX 2: Enable Gradient Checkpointing on Phi-3
        self.phi3.gradient_checkpointing_enable()

        self.hf_device_map = self.phi3.hf_device_map

        # Infer hidden dims
        clip_hidden = self.clip.config.hidden_size
        llm_hidden  = self.phi3.config.hidden_size
        logger.info("Projector: %d → %d → %d (FP32)", clip_hidden, proj_hidden, llm_hidden)

        # ── Projector (trainable, FP32) ──────────────────────────────────────
        self.projector = MLPProjector(clip_hidden, llm_hidden, proj_hidden)
        self.projector = self.projector.to("cuda:0").float()

    def get_trainable_params(self):
        return [p for p in self.projector.parameters() if p.requires_grad]

    def forward(
        self,
        pixel_values:   torch.Tensor,   # [B, 3, 224, 224]
        input_ids:      torch.Tensor,   # [B, L]
        attention_mask: torch.Tensor,   # [B, L]
        labels:         Optional[torch.Tensor] = None,  # [B, L]
    ):
        # ── 1. Vision encoding (FP16 on GPU 0) ───────────────────────────────
        pixel_values = pixel_values.to("cuda:0", dtype=torch.float16)
        with torch.no_grad():
            vision_out = self.clip(pixel_values=pixel_values)
        clip_feats = vision_out.last_hidden_state  # [B, seq, clip_hidden]

        # ── 2. Projector → LLM embedding space (FP32 on GPU 0) ───────────────
        proj_feats = self.projector(clip_feats)    # [B, seq, llm_hidden], FP32 on cuda:0

        # ── 3. Cross-device tensor logic ──────────────────────────────────────
        embed_layer  = self.phi3.get_input_embeddings()
        embed_device = embed_layer.weight.device

        input_ids_on_embed = input_ids.to(embed_device)
        with torch.no_grad():
            text_embeds = embed_layer(input_ids_on_embed)  # [B, L, llm_hidden]

        text_embeds = text_embeds.to(proj_feats.device).to(proj_feats.dtype)

        # ── 4. Prepend visual tokens to text tokens ───────────────────────────
        inputs_embeds = torch.cat([proj_feats, text_embeds], dim=1)

        vis_len = proj_feats.size(1)
        vis_attn = torch.ones(
            attention_mask.size(0), vis_len,
            dtype=attention_mask.dtype,
            device=embed_device
        )
        attention_mask_on_embed = attention_mask.to(embed_device)
        full_attention_mask = torch.cat([vis_attn, attention_mask_on_embed], dim=1)

        if labels is not None:
            vis_labels = torch.full(
                (labels.size(0), vis_len), -100,
                dtype=labels.dtype, device=embed_device
            )
            labels_on_embed = labels.to(embed_device)
            full_labels = torch.cat([vis_labels, labels_on_embed], dim=1)
        else:
            full_labels = None

        inputs_embeds = inputs_embeds.to(embed_device).to(embed_layer.weight.dtype)

        # ── 5. Phi-3 forward pass ─────────────────────────────────────────────
        outputs = self.phi3(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attention_mask,
            labels=full_labels,
            return_dict=True,
            # <--- FIX 3: Disable KV Cache during training (required for checkpointing)
            use_cache=False 
        )
        return outputs

## 7. HuggingFace Trainer Wrapper

We wrap `LLaVAStyleModel` inside a thin `nn.Module` that conforms to the HF Trainer API  
(i.e., returns a `loss` attribute when labels are present).

In [9]:
class LLaVATrainerWrapper(nn.Module):
    """Thin HF Trainer-compatible wrapper around LLaVAStyleModel."""

    def __init__(self, base_model: LLaVAStyleModel):
        super().__init__()
        self.model = base_model
        
        self.is_model_parallel = True
        self.is_parallelizable = True
        
        self.hf_device_map = self.model.hf_device_map

    # <--- FIX: Intercept the Trainer's gradient checkpointing commands
    def gradient_checkpointing_enable(self, **kwargs):
        # We already explicitly enabled this on the sub-models in Cell 6.
        # This dummy method just satisfies the Trainer so it doesn't crash.
        pass

    def gradient_checkpointing_disable(self, **kwargs):
        pass

    def forward(
        self,
        pixel_values:   torch.Tensor,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        labels:         Optional[torch.Tensor] = None,
        **kwargs,
    ):
        # Returns the native CausalLMOutput with loss and logits intact
        return self.model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

## 8. Callbacks — Disk Saver & Hub Push

In [10]:
class KaggleDiskSaverCallback(TrainerCallback):
    """
    Deletes the PREVIOUS checkpoint directory BEFORE saving the current one
    to prevent Kaggle's 19.5 GB disk limit from being exceeded.
    """

    def __init__(self, output_dir: str):
        self.output_dir = output_dir
        self._last_ckpt: Optional[str] = None

    def on_save(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        if self._last_ckpt and os.path.isdir(self._last_ckpt):
            logger.info("[DiskSaver] Removing old checkpoint: %s", self._last_ckpt)
            shutil.rmtree(self._last_ckpt)
        ckpt_dir = os.path.join(self.output_dir, f"checkpoint-{state.global_step}")
        self._last_ckpt = ckpt_dir
        return control


class HubPushCallback(TrainerCallback):
    """
    Pushes:
      - Projector weights (state_dict)
      - Training loss CSV
      - Evaluation metrics JSON
    to the Hugging Face Hub at the end of every epoch.
    """

    def __init__(self, hub_model_id: str, hf_token: str, output_dir: str):
        self.hub_model_id = hub_model_id
        self.hf_token     = hf_token
        self.output_dir   = output_dir
        if hf_token:
            login(token=hf_token)
            self.api = HfApi()
        else:
            logger.warning("[HubPush] HF_TOKEN not set — hub pushes disabled.")
            self.api = None

    def on_epoch_end(
        self,
        args,
        state: TrainerState,
        control: TrainerControl,
        model=None,
        **kwargs,
    ):
        if self.api is None or model is None:
            return control

        epoch = int(state.epoch) if state.epoch else 0
        logger.info("[HubPush] Epoch %d — pushing to %s", epoch, self.hub_model_id)

        try:
            # Save projector weights
            proj_path = os.path.join(self.output_dir, f"projector_epoch{epoch}.pt")
            torch.save(model.model.projector.state_dict(), proj_path)
            self.api.upload_file(
                path_or_fileobj=proj_path,
                path_in_repo=f"projector_epoch{epoch}.pt",
                repo_id=self.hub_model_id,
                token=self.hf_token,
                repo_type="model",
            )

            # Push loss CSV
            if os.path.isfile(LOSS_CSV_PATH):
                self.api.upload_file(
                    path_or_fileobj=LOSS_CSV_PATH,
                    path_in_repo="train_loss_log.csv",
                    repo_id=self.hub_model_id,
                    token=self.hf_token,
                    repo_type="model",
                )

            # Push log history as JSON
            metrics_path = os.path.join(self.output_dir, "log_history.json")
            with open(metrics_path, "w") as f:
                json.dump(state.log_history, f, indent=2)
            self.api.upload_file(
                path_or_fileobj=metrics_path,
                path_in_repo="log_history.json",
                repo_id=self.hub_model_id,
                token=self.hf_token,
                repo_type="model",
            )
            logger.info("[HubPush] Epoch %d push complete.", epoch)
        except Exception as e:
            logger.error("[HubPush] Failed: %s", e)
        return control


class LossLoggerCallback(TrainerCallback):
    """Appends training loss to a CSV file at every logging step."""

    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["epoch", "step", "train_loss", "eval_loss"])

    def on_log(self, args, state: TrainerState, control: TrainerControl, logs=None, **kwargs):
        if logs is None:
            return control
        with open(self.csv_path, "a", newline="") as f:
            csv.writer(f).writerow([
                round(state.epoch, 4) if state.epoch else "",
                state.global_step,
                logs.get("loss", logs.get("train_loss", "")),
                logs.get("eval_loss", ""),
            ])
        return control

## 9. Build Models, Tokenizer & Datasets

In [11]:
# ── Processors / Tokenizer ────────────────────────────────────────────────────
logger.info("Loading CLIP image processor...")
clip_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)

logger.info("Loading Phi-3 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    PHI3_MODEL_ID, 
    trust_remote_code=False,  
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Model ─────────────────────────────────────────────────────────────────────
logger.info("Building LLaVA-style model...")
base_model     = LLaVAStyleModel()
trainer_model  = LLaVATrainerWrapper(base_model)

# ── ULTIMATE DATAPARALLEL KILL-SWITCH ─────────────────────────────────────────
# By forcefully setting these attributes on the instantiated objects in memory, 
# the Hugging Face Trainer absolutely cannot ignore them during its setup phase.
base_model.is_model_parallel = True
trainer_model.is_model_parallel = True

trainable = sum(p.numel() for p in base_model.get_trainable_params())
total     = sum(p.numel() for p in base_model.parameters())
logger.info("Trainable params: %s / %s (%.2f%%)",
            f"{trainable:,}", f"{total:,}", 100 * trainable / total)

# ── Datasets ──────────────────────────────────────────────────────────────────
logger.info("Building datasets...")

# 90/10 train-val split of synthetic_train
random.shuffle(synthetic_train)
split_idx   = int(0.9 * len(synthetic_train))
train_split = synthetic_train[:split_idx]
val_split   = synthetic_train[split_idx:]

train_dataset = VizWizLFDataset(train_split, clip_processor, tokenizer, IMAGE_PATH_MAP)
val_dataset   = VizWizLFDataset(val_split,   clip_processor, tokenizer, IMAGE_PATH_MAP)

logger.info("Train: %d | Val: %d | Expert holdout: %d",
            len(train_dataset), len(val_dataset), len(expert_holdout))

PAD_ID = tokenizer.pad_token_id or 0
collator = lambda features: llava_data_collator(features, pad_token_id=PAD_ID)

2026-05-02 07:02:35,193 INFO Loading CLIP image processor...
2026-05-02 07:02:35,348 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-02 07:02:35,415 INFO HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-02 07:02:35,430 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/preprocessor_config.json "HTTP/1.1 200 OK"
2026-05-02 07:02:35,446 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/preprocessor_config.json "HTTP/1.1 200 OK"


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

2026-05-02 07:02:35,460 INFO Loading Phi-3 tokenizer...
2026-05-02 07:02:35,522 INFO HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-02 07:02:35,523 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-02 07:02:35,539 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/config.json "HTTP/1.1 200 OK"
2026-05-02 07:02:35,556 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

2026-05-02 07:02:35,637 INFO HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-02 07:02:35,652 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-02 07:02:35,668 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/tokenizer_config.json "HTTP/1.1 200 OK"


...

2026-05-02 07:02:44,952 INFO HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/f39ac1d28e925b323eae81227eaba4464caced4e/model-00002-of-00002.safetensors "HTTP/1.1 302 Found"
2026-05-02 07:02:45,029 INFO HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/f39ac1d28e925b323eae81227eaba4464caced4e/model-00001-of-00002.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

2026-05-02 07:03:25,253 INFO HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-02 07:03:25,270 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/generation_config.json "HTTP/1.1 200 OK"
2026-05-02 07:03:25,286 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3-mini-4k-instruct/f39ac1d28e925b323eae81227eaba4464caced4e/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

2026-05-02 07:03:25,365 INFO Projector: 1024 → 2048 → 3072 (FP32)
2026-05-02 07:03:25,433 INFO Trainable params: 8,393,728 / 4,132,653,056 (0.20%)
2026-05-02 07:03:25,434 INFO Building datasets...
2026-05-02 07:03:25,439 INFO Train: 3236 | Val: 360 | Expert holdout: 600


## 10. Training — HuggingFace Trainer

In [12]:
# ── Training Arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, # KEEP AT 1
    per_device_eval_batch_size=BATCH_SIZE,  # KEEP AT 1
    gradient_accumulation_steps=GRAD_ACCUM, 
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    
    # <--- FIX 4: Use 8-bit optimizer to drastically cut VRAM usage
    optim="paged_adamw_8bit", 
    
    # <--- FIX 5: Enforce gradient checkpointing at the Trainer level
    gradient_checkpointing=True,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=20,
    dataloader_num_workers=NUM_WORKERS,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    dataloader_persistent_workers=True,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

training_args._n_gpu = 1

# ── Callbacks ─────────────────────────────────────────────────────────────────
disk_saver_cb  = KaggleDiskSaverCallback(OUTPUT_DIR)
loss_logger_cb = LossLoggerCallback(LOSS_CSV_PATH)
hub_push_cb    = HubPushCallback(HUB_MODEL_ID, HF_TOKEN, OUTPUT_DIR)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=trainer_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=[disk_saver_cb, loss_logger_cb, hub_push_cb],
)

logger.info("Starting Single-Stage SFT training...")
train_result = trainer.train()
logger.info("Training complete: %s", train_result.metrics)

# Save final projector weights
final_proj_path = os.path.join(OUTPUT_DIR, "projector_final.pt")
torch.save(base_model.projector.state_dict(), final_proj_path)
logger.info("Final projector saved to: %s", final_proj_path)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
2026-05-02 07:03:25,547 INFO HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
2026-05-02 07:03:25,557 INFO Starting Single-Stage SFT training...


Epoch,Training Loss,Validation Loss
1,12.223898,1.518911
2,11.799557,1.472313
3,11.296147,1.462829


2026-05-02 10:51:50,359 INFO [HubPush] Epoch 1 — pushing to Abdullah-Khan-Niazi/vizwiz-lf-llava-projector
2026-05-02 10:51:50,673 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/preupload/main "HTTP/1.1 200 OK"
2026-05-02 10:51:50,758 INFO HTTP Request: POST https://huggingface.co/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector.git/info/lfs/objects/batch "HTTP/1.1 200 OK"
2026-05-02 10:51:50,825 INFO HTTP Request: GET https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/xet-write-token/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

2026-05-02 14:49:31,142 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/commit/main "HTTP/1.1 200 OK"
2026-05-02 14:49:31,235 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/preupload/main "HTTP/1.1 200 OK"
2026-05-02 14:49:31,560 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/commit/main "HTTP/1.1 200 OK"
2026-05-02 14:49:31,648 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/preupload/main "HTTP/1.1 200 OK"
2026-05-02 14:49:31,921 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/commit/main "HTTP/1.1 200 OK"
2026-05-02 14:49:31,922 INFO [HubPush] Epoch 2 push complete.
2026-05-02 18:47:05,136 INFO [HubPush] Epoch 3 — pushing to Abdullah-Khan-Niazi/vizwiz-lf-llava-projector
2026-05-02 18:47:05,520 INFO HTTP Re

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

2026-05-02 18:47:07,980 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/commit/main "HTTP/1.1 200 OK"
2026-05-02 18:47:08,113 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/preupload/main "HTTP/1.1 200 OK"
2026-05-02 18:47:08,536 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/commit/main "HTTP/1.1 200 OK"
2026-05-02 18:47:08,635 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/preupload/main "HTTP/1.1 200 OK"
2026-05-02 18:47:09,004 INFO HTTP Request: POST https://huggingface.co/api/models/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/commit/main "HTTP/1.1 200 OK"
2026-05-02 18:47:09,005 INFO [HubPush] Epoch 3 push complete.
2026-05-02 18:56:09,896 INFO Training complete: {'train_runtime': 42755.732, 'train_samples_per_second': 0.227, 'train_steps_per_second': 0.00

## 11. Final Hub Push — All Artifacts

In [ ]:
if HF_TOKEN:
    api = HfApi()
    upload_files = [
        (final_proj_path,           "projector_final.pt"),
        (LOSS_CSV_PATH,             "train_loss_log.csv"),
        (EXPERT_HOLDOUT_PATH,       "expert_holdout.json"),
        (preds_path,                "expert_holdout_predictions.json"),
        (metrics_path,              "nlg_metrics.json"),
    ]
    if judge_scores:
        upload_files.append((judge_path, "judge_scores.json"))

    for local_path, repo_path in upload_files:
        if os.path.isfile(local_path):
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=repo_path,
                repo_id=HUB_MODEL_ID,
                token=HF_TOKEN,
                repo_type="model",
            )
            logger.info("Uploaded: %s → %s", local_path, repo_path)

    logger.info("All artifacts pushed to HF Hub: https://huggingface.co/%s", HUB_MODEL_ID)
else:
    logger.info("HF_TOKEN not set — skipping final hub push.")

print("\n✅ Notebook 2 (LLaVA-Style Projector SFT) complete.")

2026-05-03 00:11:46,513 INFO Uploaded: kaggle/output/projector_final.pt → https://huggingface.co/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/projector_final.pt
2026-05-03 00:11:46,515 INFO Uploaded: kaggle/output/train_loss_log.csv → https://huggingface.co/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/train_loss_log.csv
2026-05-03 00:11:46,517 INFO Uploaded: kaggle/output/expert_holdout.json → https://huggingface.co/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector/expert_holdout.json
2026-05-03 00:11:46,519 INFO All artifacts pushed to HF Hub: https://huggingface.co/Abdullah-Khan-Niazi/vizwiz-lf-llava-projector



✅ Notebook 2 (LLaVA-Style Projector SFT) complete.
